# 01 - Create S3 buckets
Creates the landing, archive and rejects buckets for the SkyPoints pipeline. Safe to re-run.

In [ ]:
import boto3
from botocore.exceptions import ClientError

In [ ]:
# ---- CONFIG: edit these two values ----
SUFFIX = "iishaan"          # must make bucket names globally unique
REGION = "ap-southeast-7"       # Thailand (opt-in region: must be enabled in your AWS account)
PROFILE = None                  # AWS named profile, or None for default credentials

PREFIX = "skypoints"
BUCKETS = {
    "landing": f"{PREFIX}-landing-{SUFFIX}",   # files Snowflake COPY INTO reads
    "archive": f"{PREFIX}-archive-{SUFFIX}",   # files moved here after a successful load
    "rejects": f"{PREFIX}-rejects-{SUFFIX}",   # rejected-record extracts from COPY INTO
}
BUCKETS

In [ ]:
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
s3 = session.client("s3", region_name=REGION)
print("Account:", session.client("sts").get_caller_identity()["Account"])

In [ ]:
def create_bucket(name, versioning=False):
    try:
        kwargs = {"Bucket": name}
        if REGION != "us-east-1":   # us-east-1 must not send a LocationConstraint
            kwargs["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
        s3.create_bucket(**kwargs)
        print("created      ", name)
    except ClientError as e:
        if e.response["Error"]["Code"] == "BucketAlreadyOwnedByYou":
            print("already exists", name)
        else:
            raise

    s3.put_public_access_block(Bucket=name, PublicAccessBlockConfiguration={
        "BlockPublicAcls": True, "IgnorePublicAcls": True,
        "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
    s3.put_bucket_encryption(Bucket=name, ServerSideEncryptionConfiguration={
        "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})
    if versioning:
        s3.put_bucket_versioning(Bucket=name, VersioningConfiguration={"Status": "Enabled"})

for kind, name in BUCKETS.items():
    create_bucket(name, versioning=(kind == "landing"))

In [ ]:
# verify
existing = {b["Name"] for b in s3.list_buckets()["Buckets"]}
for name in BUCKETS.values():
    print(("OK      " if name in existing else "MISSING ") + name)